UTS 02- Regresi

# Cell 1: Instalasi Dependencies

In [1]:
# Install library yang diperlukan (Optuna untuk tuning, MLFlow untuk tracking, LIME untuk interpretasi)
!pip install -q optuna mlflow lime lightgbm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 8.8 MB/s eta 0:00:00


# Cell 2: Import Library & Persiapan Awal
Di sini kita mengatur environment dan memastikan MLFlow menyimpan log di dalam direktori kerja Kaggle agar tidak hilang saat dijalankan di background.

In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Evaluasi
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Machine Learning
import lightgbm as lgb

# Tuning, Tracking & Interpretasi
import optuna
import mlflow
import lime
import lime.lime_tabular

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RANDOM_STATE = 42

# Cell 3: Memuat Dataset
Karena dataset tidak memiliki header, kita mendefinisikan kolom pertama sebagai target (Tahun Rilis) dan sisanya sebagai fitur. Mengingat dataset ini berisi fitur karakteristik sinyal audio seperti timbre, nilai-nilai numerik ini sebenarnya merepresentasikan ekstraksi frekuensi, pitch, dan amplitudo—mirip dengan bagaimana parameter diatur pada 32-band equalizer untuk memetakan frekuensi instrumen atau memotong sibilance pada vokal, hanya saja dalam format yang diproses langsung oleh algoritma.

In [3]:
# Path dataset di Kaggle
DATA_PATH = "/kaggle/input/datasets/farisaryaputra/dataset-uts/midterm-regresi-dataset.csv"

# Load data tanpa header
df = pd.read_csv(DATA_PATH, header=None)

# Membuat nama kolom: kolom 0 = target, sisanya = feature_1, feature_2, dst.
num_features = df.shape[1] - 1
column_names = ['target'] + [f'feature_{i}' for i in range(1, num_features + 1)]
df.columns = column_names

print(f"Dimensi Data: {df.shape}")
display(df.head(3))

Dimensi Data: (515345, 91)


,target,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90
0,2001,49.94357,21.47114,73.07750,8.74861,-17.40628,-13.09905,-25.01202,-12.23257,7.83089,-2.46783,3.32136,-2.31521,10.20556,611.10913,951.08960,698.11428,408.98485,383.70912,326.51512,238.11327,251.42414,187.17351,100.42652,179.19498,-8.41558,-317.87038,95.86266,48.10259,-95.66303,-18.06215,1.96984,34.42438,11.72670,1.36790,7.79444,-0.36994,-133.67852,-83.26165,-37.29765,73.04667,-37.36684,-3.13853,-24.21531,-13.23066,15.93809,-18.60478,82.15479,240.57980,-10.29407,31.58431,-25.38187,-3.90772,13.29258,41.55060,-7.26272,-21.00863,105.50848,64.29856,26.08481,-44.59110,-8.30657,7.93706,-10.73660,-95.44766,-82.03307,-35.59194,4.69525,70.95626,28.09139,6.02015,-37.13767,-41.12450,-8.40816,7.19877,-8.60176,-5.90857,-12.32437,14.68734,-54.32125,40.14786,13.01620,-54.40548,58.99367,15.37344,1.11144,-23.08793,68.40795,-1.82223,-27.46348,2.26327
1,2001,48.73215,18.42930,70.32679,12.94636,-10.32437,-24.83777,8.76630,-0.92019,18.76548,4.59210,2.21920,0.34006,44.38997,2056.93836,605.40696,457.41175,777.15347,415.64880,746.47775,366.45320,317.82946,273.07917,141.75921,317.35269,19.48271,-65.25496,162.75145,135.00765,-96.28436,-86.87955,17.38087,45.90742,32.49908,-32.85429,45.10830,26.84939,-302.57328,-41.71932,-138.85034,202.18689,-33.44277,195.04749,-16.93235,-1.09168,-25.38061,-12.19034,-125.94783,121.74212,136.67075,41.18157,28.55107,1.52298,70.99515,-43.63073,-42.55014,129.82848,79.95420,-87.14554,-45.75446,-65.82100,-43.90031,-19.45705,12.59163,-407.64130,42.91189,12.15850,-88.37882,42.25246,46.49209,-30.17747,45.98495,130.47892,13.88281,-4.00055,17.85965,-18.32138,-87.99109,14.37524,-22.70119,-58.81266,5.66812,-19.68073,33.04964,42.87836,-9.90378,-32.22788,70.49388,12.04941,58.43453,26.92061
2,2001,50.95714,31.85602,55.81851,13.41693,-6.57898,-18.54940,-3.27872,-2.35035,16.07017,1.39518,2.73553,0.82804,7.46586,699.54544,1016.00954,594.06748,355.73663,507.39931,387.69910,287.15347,112.37152,161.68928,144.14353,199.29693,-4.24359,-297.00587,-148.36392,-7.94726,-18.71630,12.77542,-25.37725,9.71410,0.13843,26.79723,6.30760,28.70107,-74.89005,-289.19553,-166.26089,13.09302,5.89085,6.89034,-10.97160,1.67565,11.43523,-7.27994,133.08169,141.86758,-56.99356,98.15952,18.50939,16.97216,24.26629,-10.50788,-8.68412,54.75759,194.74034,7.95966,-18.22685,0.06463,-2.63069,26.02561,1.75729,-262.36917,-233.60089,-2.50502,-12.14279,81.37617,2.07554,-1.82381,183.65292,22.64797,-39.98887,43.37381,-31.56737,-4.88840,-36.53213,-23.94662,-84.19275,66.00518,3.03800,26.05866,-50.92779,10.93792,-0.07568,43.20130,-115.00698,-0.05859,39.67068,-0.66345


# Cell 4: Data Cleaning & Outlier Handling
Kita tangani nilai kosong dan membatasi nilai outlier yang bisa mengacaukan regresi menggunakan rentang Interquartile (IQR).

In [4]:
# 1. Menangani Missing Values (menggunakan median agar lebih kebal terhadap nilai ekstrem)
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# 2. Outlier Handling dengan metode IQR Clipping
# Mengganti nilai ekstrem dengan nilai batas atas/bawah agar data tidak banyak terbuang
feature_cols = [col for col in df_imputed.columns if col != 'target']

df_clean = df_imputed.copy()
for col in feature_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip (batasi) nilai outlier
    df_clean[col] = np.clip(df_clean[col], lower_bound, upper_bound)

print("Handling missing values & outliers selesai.")

Handling missing values & outliers selesai.


# Cell 5: Feature Engineering & Split Data
Mencari fitur yang punya korelasi terhadap tahun rilis. Fitur yang benar-benar tidak berkorelasi (mendekati 0) akan dibuang.

In [5]:
# Menghitung korelasi absolut dengan target
correlations = df_clean.corr()['target'].abs()

# Seleksi fitur yang memiliki korelasi di atas threshold (misal: 0.05)
CORR_THRESHOLD = 0.05
selected_features = correlations[correlations > CORR_THRESHOLD].index.tolist()
selected_features.remove('target') # Pisahkan target dari daftar fitur

print(f"Fitur awal: {len(feature_cols)} | Fitur terpilih: {len(selected_features)}")

# Pisahkan X dan y
X = df_clean[selected_features]
y = df_clean['target']

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Scaling menggunakan RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Fitur awal: 90 | Fitur terpilih: 42


# Cell 6: Hyperparameter Tuning (Optuna) & Tracking (MLFlow)
Kita akan mencari parameter terbaik untuk model LightGBM menggunakan Optuna, dan semua proses uji cobanya akan dicatat secara otomatis oleh MLFlow.

In [6]:
import joblib
from lightgbm import early_stopping

# Setup MLFlow
mlflow.set_experiment("Song_Release_Year_Prediction")

# Variabel global untuk menyimpan skor terbaik sementara
best_rmse_so_far = float('inf')
MODEL_PATH = '/kaggle/working/best_lgbm_model.pkl'
DB_PATH = 'sqlite:////kaggle/working/optuna_study.db'

def objective(trial):
    global best_rmse_so_far
    
    with mlflow.start_run(nested=True):
        # 1. Ruang pencarian parameter
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'verbosity': -1,
            'random_state': RANDOM_STATE,
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000), # Bisa diset tinggi karena ada early stopping
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100)
        }
        
        model = lgb.LGBMRegressor(**params)
        
        # 2. Fit model dengan Early Stopping
        # Kita pisahkan sebagian kecil data train untuk validasi early stopping agar tidak bocor ke data test
        X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.1, random_state=RANDOM_STATE)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[early_stopping(stopping_rounds=30, verbose=False)] # Berhenti jika 30 iterasi tidak membaik
        )
        
        # 3. Prediksi ke Data Test Utama
        preds = model.predict(X_test_scaled)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        
        # 4. CACHING: Simpan model JIKA mendapatkan skor RMSE yang lebih kecil (lebih baik)
        if rmse < best_rmse_so_far:
            best_rmse_so_far = rmse
            # Simpan fisik model ke folder Kaggle
            joblib.dump(model, MODEL_PATH)
            print(f"Bintang baru! Model terbaik disimpan dengan RMSE: {rmse:.4f}")
            
        # Log parameter dan metrik ke MLFlow
        mlflow.log_params(params)
        mlflow.log_metric("rmse", rmse)
        
        return rmse

print("Memulai Optuna dengan sistem Caching & Early Stopping...")

# 5. CACHING OPTUNA: Menggunakan SQLite agar bisa di-resume
study = optuna.create_study(
    direction='minimize',
    study_name='lgbm_regression_study',
    storage=DB_PATH,        # Simpan log trial ke database
    load_if_exists=True     # JIKA Kaggle mati, dia akan membaca database ini dan melanjutkan!
)

# Jalankan tuning (misal 20 trial)
study.optimize(objective, n_trials=20)

print(f"\nTuning Selesai. Hyperparameter Terbaik: {study.best_params}")
print(f"Model terbaik sudah di-cache secara otomatis di: {MODEL_PATH}")

2026/05/12 20:34:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/12 20:34:32 INFO mlflow.store.db.utils: Updating database tables
2026/05/12 20:34:35 INFO mlflow.tracking.fluent: Experiment with name 'Song_Release_Year_Prediction' does not exist. Creating a new experiment.


Memulai Optuna dengan sistem Caching & Early Stopping...


[I 2026-05-12 20:34:36,477] A new study created in RDB with name: lgbm_regression_study
[I 2026-05-12 20:35:00,699] Trial 0 finished with value: 9.313214209614115 and parameters: {'n_estimators': 743, 'learning_rate': 0.07276919655273807, 'max_depth': 3, 'num_leaves': 74}. Best is trial 0 with value: 9.313214209614115.


Bintang baru! Model terbaik disimpan dengan RMSE: 9.3132


[I 2026-05-12 20:35:58,344] Trial 1 finished with value: 9.28576660989761 and parameters: {'n_estimators': 879, 'learning_rate': 0.010872827983624659, 'max_depth': 6, 'num_leaves': 57}. Best is trial 1 with value: 9.28576660989761.


...

Bintang baru! Model terbaik disimpan dengan RMSE: 9.1204


[I 2026-05-12 20:41:13,049] Trial 10 finished with value: 9.206110596782494 and parameters: {'n_estimators': 256, 'learning_rate': 0.03724765515159807, 'max_depth': 12, 'num_leaves': 97}. Best is trial 9 with value: 9.120448937575805.
[I 2026-05-12 20:41:41,126] Trial 11 finished with value: 9.198114461690114 and parameters: {'n_estimators': 481, 'learning_rate': 0.044553497510033506, 'max_depth': 8, 'num_leaves': 51}. Best is trial 9 with value: 9.120448937575805.
[I 2026-05-12 20:42:17,561] Trial 12 finished with value: 9.191254143669488 and parameters: {'n_estimators': 490, 'learning_rate': 0.028665591593100344, 'max_depth': 8, 'num_leaves': 81}. Best is trial 9 with value: 9.120448937575805.
[I 2026-05-12 20:42:32,647] Trial 13 finished with value: 9.225735919809527 and parameters: {'n_estimators': 983, 'learning_rate': 0.19616845755481066, 'max_depth': 9, 'num_leaves': 47}. Best is trial 9 with value: 9.120448937575805.
[I 2026-05-12 20:42:51,647] Trial 14 finished with value: 9.2

Bintang baru! Model terbaik disimpan dengan RMSE: 9.1007


[I 2026-05-12 20:45:03,544] Trial 17 finished with value: 9.103836142830334 and parameters: {'n_estimators': 919, 'learning_rate': 0.04785543586626176, 'max_depth': 9, 'num_leaves': 100}. Best is trial 16 with value: 9.100713452941811.
[I 2026-05-12 20:45:57,666] Trial 18 finished with value: 9.125242357999234 and parameters: {'n_estimators': 993, 'learning_rate': 0.03604915778413895, 'max_depth': 9, 'num_leaves': 80}. Best is trial 16 with value: 9.100713452941811.
[I 2026-05-12 20:46:52,267] Trial 19 finished with value: 9.205966925623638 and parameters: {'n_estimators': 914, 'learning_rate': 0.024209186501092574, 'max_depth': 6, 'num_leaves': 99}. Best is trial 16 with value: 9.100713452941811.



Tuning Selesai. Hyperparameter Terbaik: {'n_estimators': 937, 'learning_rate': 0.06739883647378261, 'max_depth': 7, 'num_leaves': 99}
Model terbaik sudah di-cache secara otomatis di: /kaggle/working/best_lgbm_model.pkl


# Cell 7: Evaluasi Model Akhir & Interpretasi (LIME)
Kita akan melatih model sekali lagi menggunakan parameter terbaik dari Optuna, menghitung metrik utama, dan menjelaskan prediksinya.

# 1. Melatih Model Final dengan best parameters
final_model = lgb.LGBMRegressor(**best_params, random_state=RANDOM_STATE, verbosity=-1)
final_model.fit(X_train_scaled, y_train)

# Prediksi menggunakan data test
y_pred = final_model.predict(X_test_scaled)

# 2. Evaluasi Metrik Regresi
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"--- Evaluasi Model Final ---")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R^2  : {r2:.4f}")

# Log final model ke MLflow
with mlflow.start_run():
    mlflow.log_params(best_params)
    mlflow.log_metrics({"final_mse": mse, "final_rmse": rmse, "final_mae": mae, "final_r2": r2})
    # mlflow.sklearn.log_model(final_model, "lightgbm_model") # Opsional: simpan model fisiknya

# 3. Interpretasi dengan LIME
# Inisialisasi explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=selected_features,
    class_names=['Year'],
    mode='regression',
    random_state=RANDOM_STATE
)

# Ambil satu sampel data dari X_test (indeks ke-0) untuk diinterpretasikan
sample_idx = 0
exp = explainer.explain_instance(
    data_row=X_test_scaled[sample_idx], 
    predict_fn=final_model.predict, 
    num_features=5 # Tampilkan 5 fitur paling berpengaruh
)

print(f"\n--- LIME Interpretation ---")
print(f"Tahun Rilis Sebenarnya: {y_test.iloc[sample_idx]}")
print(f"Prediksi Model: {y_pred[sample_idx]:.2f}")

# Menampilkan grafik interpretasi di dalam Kaggle/Jupyter
exp.show_in_notebook(show_table=True, show_all=False)